In [1]:
import subprocess
import os
import time

# Get the kernel PID
mgr_parent_pid = os.getpid()
print(f"Kernel PID: {mgr_parent_pid}")

# Define the strace command
strace_cmd = [
    "sudo", "-S",  # -S reads password from stdin
    "strace",
    "-qqq",  
    "-r",   
    "-z",  
    "-f",    # Follow forks/threads
    "-o", "/home/admin/shahadat/floability-env.cli/strace_manager.txt", 
    "-p", str(mgr_parent_pid),
]

# Start strace in the background with sudo
mgr_strace = subprocess.Popen(
    strace_cmd,
    stdin=subprocess.PIPE,  # Enable stdin for password
    stdout=subprocess.PIPE,  
    stderr=subprocess.PIPE, 
    preexec_fn=os.setsid
)

# Send the sudo password
password = "cdm_depaul"  # Replace with your actual password
mgr_strace.stdin.write(password.encode() + b"\n")  # Encode and add newline
mgr_strace.stdin.flush() 

print("strace is running in the background. Execute your cells now!")

time.sleep(1)
mgr_strace.stdin.close()

from functools import wraps
def worker(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with open('_tmp.txt', 'w') as f: 
            f.write('hello1')
            result = func(*args, **kwargs)
            f.write('hello2')
            print(result)
            return result
    return wrapper


Kernel PID: 115992
strace is running in the background. Execute your cells now!


In [2]:
@worker
def multiply_pair(A, B):
    import numpy as np  # Only on the worker
    A_np = np.array(A, dtype=float)
    B_np = np.array(B, dtype=float)
    C_np = A_np @ B_np
    return C_np.tolist()  # convert back to nested list


In [3]:
import os

manager_name = api_key = os.environ.get("VINE_MANAGER_NAME")
print(f"Manager name: {manager_name}")

Manager name: None


In [4]:
import random
import ndcctools.taskvine as vine

N = 50      # 50x50 matrices
num_pairs = 10  # We'll do 10 pairs for demonstration

A_list = []
B_list = []

for _ in range(num_pairs):
    A_mat = [[random.random() for _ in range(N)] for _ in range(N)]
    B_mat = [[random.random() for _ in range(N)] for _ in range(N)]
    A_list.append(A_mat)
    B_list.append(B_mat)

m = vine.Manager([9123,9150], name=manager_name)

print(f"[manager] Listening on port {m.port}")

tasks_map = {}
results = [None]*num_pairs


workers = vine.Factory(manager=m)
workers.max_workers = 1
workers.min_workers = 1
workers.cores = 1
workers.memory = 1000
workers.disk = 1000


for i in range(num_pairs):
    task = vine.PythonTask(multiply_pair, A_list[i], B_list[i])
    t_id = m.submit(task)
    tasks_map[t_id] = i
    print(f"[manager] Submitted multiplication for pair index {i}.")

print("[manager] Waiting for tasks to complete...")

with workers:
    while not m.empty():
        done_task = m.wait(5)
        if done_task:
            idx = tasks_map[done_task.id]
            if done_task.successful():
                C = done_task.output
                results[idx] = C
                print(f"[manager] Pair {idx} -> result row0 length = {len(C[0])}")
            else:
                print(f"[manager] Task for pair {idx} failed: {done_task.result}")
    
    print("\n[manager] All tasks done.")

[manager] Listening on port 9123
[manager] Submitted multiplication for pair index 0.
[manager] Submitted multiplication for pair index 1.
[manager] Submitted multiplication for pair index 2.
[manager] Submitted multiplication for pair index 3.
[manager] Submitted multiplication for pair index 4.
[manager] Submitted multiplication for pair index 5.
[manager] Submitted multiplication for pair index 6.
[manager] Submitted multiplication for pair index 7.
[manager] Submitted multiplication for pair index 8.
[manager] Submitted multiplication for pair index 9.
[manager] Waiting for tasks to complete...
[manager] Pair 9 -> result row0 length = 50
[manager] Pair 2 -> result row0 length = 50
[manager] Pair 3 -> result row0 length = 50
[manager] Pair 0 -> result row0 length = 50
[manager] Pair 4 -> result row0 length = 50
[manager] Pair 1 -> result row0 length = 50
[manager] Pair 7 -> result row0 length = 50
[manager] Pair 6 -> result row0 length = 50
[manager] Pair 5 -> result row0 length = 5

In [5]:
import signal

try:
    # Terminate the process group (sudo and strace)
    os.killpg(os.getpgid(mgr_strace.pid), signal.SIGTERM)
    mgr_strace.wait(timeout=5)
    print(f"strace group terminated, exit code: {mgr_strace.returncode}")
except subprocess.TimeoutExpired:
    print("Graceful termination timed out, forcing kill")
    # Kill the process group
    os.killpg(os.getpgid(mgr_strace.pid), signal.SIGKILL)
    mgr_strace.wait()
    # Double-check strace is gone
    #subprocess.run(f"sudo pkill -f 'strace -p {mgr_parent_pid}'", shell=True)
    print("strace group killed")


strace group terminated, exit code: -15
